In [2]:
# Why Text Splitting 
# for text with more length than model context length
# Better Results wih small text
# Faster Execusion
# for sementic search ( embedding quality is better in smaller text)
# For summrisation also better 
# memory efficinent and good for parallelisation

In [3]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict , Annotated , List , Optional
import warnings
warnings.filterwarnings("ignore")
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser , JsonOutputParser 

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

parser = StrOutputParser()

llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash" , api_key= GOOGLE_API_KEY)
llm_gemini.invoke("who is father of india").content

'Mahatma Gandhi is widely considered the "Father of India" for his pivotal role in the Indian independence movement.'

In [18]:
# Length Based text splitting
# super fast and simple
# not sementic meaning, no grammer , not meaning in the chunk
# Hennce not use that much

# chunk_overlap --> how much overlap between two chunks
# Why chunk_overlap --> idea is the two chunks have similar contexts 
# Normally --> 10-20% overlap is good for RAG based Solutions 

from langchain.text_splitter import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

loader = TextLoader(file_path="Data\sample_text.txt" , encoding= "utf-8")
docu = loader.load()


splitter = CharacterTextSplitter(
    chunk_size = 100 , 
    chunk_overlap = 0,
    separator= " "  
    )

results = splitter.split_documents(docu)
for id , chunk in enumerate(results):
    print(f"chunk number {id+1} is : {chunk.page_content.replace("\n" , "")}")

chunk number 1 is : Data Scientist: 🕵️‍♂️ Finds insights in data, builds models. AI Engineer: 🛠️ Deploys those models at
chunk number 2 is : scale!Think: Data Scientist is the architect, AI Engineer is the construction crew. Both build the
chunk number 3 is : future! 🚀 #DataScience #AI #Tech #Jobs #Career## Data Scientist vs. AI Engineer: Decoding the Buzz!
chunk number 4 is : 🤖🧠Ever get these two roles mixed up? You're not alone! Data Scientist and AI Engineer are both hot
chunk number 5 is : careers, but they tackle different sides of the AI coin.Think of it this way: **Data Scientists
chunk number 6 is : are the visionaries. They *discover* insights from data, building models to predict future trends
chunk number 7 is : and answer critical business questions.** They're all about asking "why" and "what."**AI Engineers
chunk number 8 is : are the builders. They *deploy* those models into real-world applications.** They're focused on
chunk number 9 is : making the AI work, scale, and i

In [ ]:
# Text Structure Based
# One the most used character splitting techniques
# "\n\n" --> para , "\n" -> line , "_" -> word

from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300 , 
    chunk_overlap = 0
) 
chunks = splitter.split_text(str(docu[0].page_content))
print(len(chunks))
print(chunks[0])
print(chunks[1])

8
Data Scientist: 🕵️‍♂️ Finds insights in data, builds models. AI Engineer: 🛠️ Deploys those models at scale!
Think: Data Scientist is the architect, AI Engineer is the construction crew. Both build the future! 🚀 #DataScience #AI #Tech #Jobs #Career
## Data Scientist vs. AI Engineer: Decoding the Buzz! 🤖🧠


In [ ]:
############ Document Structure Based Splitting ####################

# We USe RecursiveCharacterTextSplitter , just the seperators are diff 
# like in markdown --> seperators are "#" , "###" , "####"
# Similar to this we can apply this in code , --> seperators will be "class" , "def"
# Can be used for code, HTML , Markdown

from langchain.text_splitter import RecursiveCharacterTextSplitter , Language

splitter = RecursiveCharacterTextSplitter.from_language(
    language= Language.PYTHON, # Have mutiple Language Support, like markdown and JS, JAVA, C , HTML
    chunk_size = 200 , 
    chunk_overlap = 0
)


loader = TextLoader(file_path="Data\sample_code.txt" , encoding= "utf-8")
docu = loader.load()

results = splitter.split_documents(docu)
print(len(results))
print(results[1])

8
page_content='class LLM:
    def __init__(self):
        print("LLM Invoked")
        
    def predict(self , prompt):
        return {"responce":f"the ans of the prompt {prompt} is Really Easy to give"}' metadata={'source': 'Data\\sample_code.txt'}


In [44]:
# Sometime both the approach fails( have 2 para , but 1 para have 2 topics while the other para have obly 1 meaning )
# Semantic Meaning Based approach
# Experimental Right Now

# Performace not that much right now

from langchain_experimental.text_splitter import SemanticChunker
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings

splitter = SemanticChunker(
    GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"),
    breakpoint_threshold_type="standard_deviation",
    breakpoint_threshold_amount=0.5
)

loader = TextLoader(file_path="Data\sample_text.txt" , encoding= "utf-8")
docu = loader.load()

results = splitter.split_documents(docu)
print(len(results))
print(results[1])


6
page_content='Both build the future! 🚀 #DataScience #AI #Tech #Jobs #Career
## Data Scientist vs. AI Engineer: Decoding the Buzz! 🤖🧠

Ever get these two roles mixed up?' metadata={'source': 'Data\\sample_text.txt'}


In [45]:
for id , chunk in enumerate(results):
    print(f"chunk number {id+1} is : {chunk.page_content.replace("\n" , "")}")

chunk number 1 is : Data Scientist: 🕵️‍♂️ Finds insights in data, builds models. AI Engineer: 🛠️ Deploys those models at scale! Think: Data Scientist is the architect, AI Engineer is the construction crew.
chunk number 2 is : Both build the future! 🚀 #DataScience #AI #Tech #Jobs #Career## Data Scientist vs. AI Engineer: Decoding the Buzz! 🤖🧠Ever get these two roles mixed up?
chunk number 3 is : You're not alone! Data Scientist and AI Engineer are both hot careers, but they tackle different sides of the AI coin. Think of it this way: **Data Scientists are the visionaries. They *discover* insights from data, building models to predict future trends and answer critical business questions.** They're all about asking "why" and "what."**AI Engineers are the builders. They *deploy* those models into real-world applications.** They're focused on making the AI work, scale, and integrate seamlessly.
chunk number 4 is : Think production-ready AI! **🔑 Key Differences:***   **Data Scientists:** Sta